# Fine-tuning GPT-2 (small) to detect Spam messages

This notebook fine-tunes the GPT-2 **124** Million parameter (pre-trained) base model on the SMS Spam Collection v.1 dataset.

- About the model: GPT-2 is a transformers model pretrained on a very large corpus of English data in a self-supervised fashion. This means it was pretrained on the raw texts only, with no humans labelling them in any way (which is why it can use lots of publicly available data) with an automatic process to generate inputs and labels from those texts. More precisely, it was trained to guess the next word in sentences. It was first released [here](https://openai.com/index/better-language-models/).

- About the dataset: The SMS Spam Collection v.1 is a public set of SMS labeled messages that have been collected for mobile phone spam research. It has one collection composed by 5,574 English, real and non-encoded messages, tagged according being legitimate (ham) or spam. Link to the [dataset](https://huggingface.co/datasets/ucirvine/sms_spam).

### Load the SMS Spam dataset

In [29]:
from datasets import load_dataset

sms_spam_dataset = load_dataset('ucirvine/sms_spam')
print(sms_spam_dataset)

DatasetDict({
    train: Dataset({
        features: ['sms', 'label'],
        num_rows: 5574
    })
})


### Performing Train-Test Split

Split the dataset into Training and Validation data. We keep **90%** of the dataset for training (fine-tuning the LLM) and the rest 10% for validating the training (fine-tuning) process. 

In [34]:
train_test_split = 0.9

shuffled_dataset = sms_spam_dataset['train'].shuffle()
train_len = int(len(shuffled_dataset) * train_test_split)

training_data = shuffled_dataset[:train_len]
validation_data = shuffled_dataset[train_len:]

In [37]:
def get_spam_non_spam_count(data):
    spam = 0
    for label in data['label']:
        if label == 0:
            spam += 1
    print(f'Spam: {spam}, Non Spam: {len(data['label']) - spam}')

print(f'Training data size: {len(training_data['sms'])}')
print(f'Validation data size: {len(validation_data['sms'])}')
get_spam_non_spam_count(training_data)
get_spam_non_spam_count(validation_data)

Training data size: 5016
Validation data size: 558
Spam: 4344, Non Spam: 672
Spam: 483, Non Spam: 75


## Tokenize the dataset

We use GPT-2 Tokenizer for tokenizing our SMS Spam dataset.

In [38]:
from transformers import GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

In [45]:
def tokenize_dataset(dataset):
    tokenized_dataset = []

    for message, label in zip(dataset['sms'], dataset['label']):
        encoded_ids = tokenizer.encode(message)
        tokenized_dataset.append((encoded_ids, label))
    
    return tokenized_dataset

In [50]:
tokenized_training_dataset = tokenize_dataset(training_data)
tokenized_validation_dataset = tokenize_dataset(validation_data)

tokenized_training_dataset[0]

([43, 2611, 78, 0, 44460, 352, 198], 0)

In [ ]:
## Load the GPT-2 Model

In [51]:
from transformers import GPT2Model

model = GPT2Model.from_pretrained('gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2393.82it/s, Materializing param=wte.weight]             
GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
model

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

We can see the final (Layer-Norm) layer has an out dimension of **768**. We need to add an additional custom layer after the LayerNorm layer which will classify the input text as **spam (0)** or **not-spam (1)**.

Before that let's generate some content from the GPT-2 model.

In [53]:
from transformers import pipeline
generator = pipeline('text-generation', model='gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1623.18it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [54]:
generator("How are you doing", max_length=30, num_return_sequences=1)

Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "How are you doing?\n\nWe're going to do this again. We're going to do this again. We're going to do this again. We are going to get better. We're going to do better.\n\nWe're going to get better. We are going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n\nWe're going to do better.\n"}]

As you can see from the above generated output from the model, it's currently an auto-complete engine which tries to complete the input text. It is not trained to classify spam and non-spam messages yet!

Moving forward, we will wrap the current GPT-2 model in our custom model class and add an output layer which will allow model to generate classification labels for spam and non-spam messages.

In [55]:
from torch import nn
import torch


class GptForSentimentAnalysis(nn.Module):
    def __init__(self):
        super(GptForSentimentAnalysis, self).__init__()

        self.gpt2 = GPT2Model.from_pretrained('gpt2') # out dimension = 768
        self.out_layer = nn.Linear(768, 2) # out dimension = 2
    
    def forward(self, input_ids):
        outputs = self.gpt2.forward(input_ids=input_ids)
        hidden_states = outputs.last_hidden_state

        # --- THE CRITICAL GPT-2 TRICK ---
        # Because GPT-2 is a causal (left-to-right) model, the meaningful context 
        # for the whole sentence accumulates at the VERY LAST token. 
        # So, we usually extract just the last token's hidden state to pass to our new layer.
        
        # Grab the last token's representation for each sequence in the batch
        # (Assuming you are padding appropriately)
        last_token_states = hidden_states[:, -1, :] # B, 1, 768

        final_output = self.out_layer(last_token_states) # B, 1, 2

        return final_output
    
    def predict(self, sms: str):
        encoded_ids = tokenizer.encode(sms)
        encoded_ids = torch.tensor([encoded_ids])

        logits = self.forward(encoded_ids)
        predicted_label = torch.argmax(logits, dim=-1)

        verdict = "Not Spam" if predicted_label == 0 else "Spam"

        return verdict

In [56]:
gpt2_sentiment_model = GptForSentimentAnalysis()
gpt2_sentiment_model

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2629.33it/s, Materializing param=wte.weight]             
GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GptForSentimentAnalysis(
  (gpt2): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (out_layer): Linear(in_features=768, out_features=2, bias=True)
)

In [69]:
f'{(sum(parameter.numel() for parameter in gpt2_sentiment_model.parameters()) / 1e6):.2f} million parameters'

'124.44 million parameters'

We have added an output layer which has an `out_feature` value of **2**.

## Now it's time to fine-tune the model. 

Since we are already working on a pre-trained GPT-2 model, we will not train the entire parameters of the model. Instead we will train the parameters of the last attention block and the custom layer only. Rest of the parameters will be frozen!

In [57]:
# Feeze the entire model
for parameter in gpt2_sentiment_model.parameters():
    parameter.requires_grad = False

# Making the last attention block trainable
for parameter in gpt2_sentiment_model.gpt2.h[-1].parameters():
    parameter.requires_grad = True

# Making the last output layer trainable as well
for parameter in gpt2_sentiment_model.out_layer.parameters():
    parameter.requires_grad = True

## Starting the Fine-tune process...

In [61]:
def get_mean(losses):
    sum = 0.0
    for loss in losses:
        sum += loss.item()
    return sum / len(losses)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

gpt2_sentiment_model.train()

epochs = 1
batch_size = 8
eval_intervals = batch_size * 50 # We log the training loss after processing 50 batches
losses = []

optimizer = torch.optim.AdamW(gpt2_sentiment_model.parameters(), lr=5e-5, weight_decay=0.1)

for i in range(epochs):
    Xb = []
    Yb = []

    for i, row in enumerate(tokenized_training_dataset):
        encoded_msg = row[0]
        label = row[1]
        Xb.append(torch.tensor(encoded_msg))
        Yb.append(torch.tensor(label))

        if (len(Xb) % batch_size) == 0:
            padded_batch = pad_sequence(Xb, batch_first=True, padding_value=0)
            logits = gpt2_sentiment_model(padded_batch)
            loss = torch.nn.functional.cross_entropy(logits, torch.tensor(Yb))
            losses.append(loss)
            loss.backward()
            optimizer.step()
            Xb = []
            Yb = []

            if (i+1)%eval_intervals == 0:
                print(f'Interval: {i+1}, Loss: {get_mean(losses)}')
                losses = []
    
    if len(Xb) > 0:
        padded_batch = pad_sequence(Xb, batch_first=True, padding_value=0)
        logits = gpt2_sentiment_model(padded_batch)
        loss = torch.nn.functional.cross_entropy(logits, torch.tensor(Yb))
        losses.append(loss)
        loss.backward()
        optimizer.step()

Let's also write a method to get the model's accuracy on the entire dataset.

In [77]:
def get_dataset_accuracy(tokenized_dataset, log_interval):
    encoded_messages = [row[0] for row in tokenized_dataset]
    labels = [row[1] for row in tokenized_dataset]
    processed = 0
    correct_predictions = 0

    for encoded_message, label in zip(encoded_messages, labels):
        with torch.no_grad():
            encoded_message = torch.tensor(encoded_message).view(1,len(encoded_message))
            logits = gpt2_sentiment_model(torch.tensor(encoded_message))
        predicted_label = torch.argmax(logits, dim=-1)
        
        if predicted_label == label:
            correct_predictions += 1
        
        if processed%log_interval == 0:
            print(f'Processed: {processed}')
        processed += 1

    print(f'{correct_predictions} / {len(encoded_messages)}')
    print(f'Accuracy: {(correct_predictions/len(encoded_messages)) * 100.0}')

In [78]:
print(f'Training dataset accuracy: {get_dataset_accuracy(tokenized_training_dataset, log_interval=500)}')

/var/folders/q7/3g65n_191jxgh71z8qlfr9th00v5bg/T/ipykernel_41421/101266929.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  logits = gpt2_sentiment_model(torch.tensor(encoded_message))


Processed: 0
Processed: 500
Processed: 1000
Processed: 1500
Processed: 2000
Processed: 2500
Processed: 3000
Processed: 3500
Processed: 4000
Processed: 4500
Processed: 5000
4925 / 5016
Accuracy: 98.18580542264753
Training dataset accuracy: None


In [79]:
print(f'Validation dataset accuracy: {get_dataset_accuracy(tokenized_validation_dataset, log_interval=100)}')

/var/folders/q7/3g65n_191jxgh71z8qlfr9th00v5bg/T/ipykernel_41421/101266929.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  logits = gpt2_sentiment_model(torch.tensor(encoded_message))


Processed: 0
Processed: 100
Processed: 200
Processed: 300
Processed: 400
Processed: 500
545 / 558
Accuracy: 97.67025089605734
Validation dataset accuracy: None


In [81]:
gpt2_sentiment_model.predict('Last a chance to win 1 Lakh, call at 5676 now!')

'Spam'